# **Structured Outputs and Error Handling**

## **What's Covered?**
1. Introduction to Structured Output
    - Introduction to Structured Output
    - Enforcing Structured Output
    - Import Syntax
2. Implementing Structured Responses
    - Step 1: Defining the Schema for response_format
    - Step 2: Agent Init with ProviderStrategy for Structured Output
    - Step 3: Agent Init with Invoking the Agent and Printing structured_response
    - Step 4: ToolStrategy for Structured Output
    - Passing Multiple Schemas
3. Error Handling with ToolStrategy
    - Multiple Structured Outputs Error
    - Schema Validation Error
    - Custom Errors (Coming Soon)

## **Introduction to Structured Output**

### **Why Structured Output**
Structured output allows agents to return data in a specific, predictable format. Instead of parsing natural language responses, you get structured data in the form of JSON objects, Pydantic models, or dataclasses that your application can directly use. 


### **Enforcing Structured Output** 
We can enforce structured output within **create_agent()** using **response_format** to return validated data in a specific schema (Pydantic/JSON) instead of free-form text.

**response_format** parameter inside **create_agent()** controls how the agent returns structured data. To apply structured output, you should either use ProviderStrategy (default) or ToolStrategy.

- **ProviderStrategy**: Some model providers support structured output natively through their APIs (e.g. OpenAI, Grok, Gemini). This is the most reliable method when available.
- **ToolStrategy**: For models that don’t support native structured output, LangChain uses tool calling to achieve the same result. This works with all models that support tool calling, which is most modern models.


### **Import Syntax**
```python
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy
```

## **Implementing Structured Responses**

### **Step 1: Defining the Schema for response_format**

In [1]:
from pydantic import BaseModel, Field

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

### **Step 2: Agent Init with ProviderStrategy for Structured Output**

Some model providers support structured output natively through their APIs (e.g. OpenAI, Grok, Gemini, Claude). This is the most reliable method when available.

```python
class ProviderStrategy(Generic[SchemaT]):
    schema: type[SchemaT]
    strict: bool | None = None
```

- **schema:** The schema defining the structured output format.
- **strict:** Optional boolean parameter to enable strict schema adherence. Supported by some providers (e.g., OpenAI and xAI). Defaults to None (disabled).

LangChain automatically uses ProviderStrategy when you pass a schema type directly to create_agent.response_format and the model supports native structured output:

```python
agent = create_agent(
    model=openai_chat_model,
    response_format=ContactInfo,  # Auto-selects ProviderStrategy
    system_prompt="""You are a helpful assistant. 
                     Be concise and accurate. 
                     You are suppose to extract contact info from the user inputs.
                     """
)
```

In [2]:
from langchain_openai import ChatOpenAI

# Setup API Key
f = open('keys/.openai_api_key.txt')
OPENAI_API_KEY = f.read()

openai_chat_model = ChatOpenAI(
    openai_api_key=OPENAI_API_KEY,
    model="gpt-4o-mini",
    temperature=0.0
)

In [3]:
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy

agent = create_agent(
    model=openai_chat_model,
    response_format=ProviderStrategy(schema=ContactInfo),
    system_prompt="""You are a helpful assistant. 
                     Be concise and accurate. 
                     You are suppose to extract contact info from the user inputs.
                     """
)

### **Step 3: Invoking the Agent and Printing structured_response**

LangChain’s **create_agent** handles structured output automatically. The user sets their desired structured output schema, and when the model generates the structured data, it’s captured, validated, and **returned in the `'structured_response'` key** of the agent’s state.

In [4]:
result = agent.invoke({
    "messages": [{"role": "human", 
                  "content": "My name is John Doe, email john@example.com, and number (555) 123-4567"}]
})

for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

My name is John Doe, email john@example.com, and number (555) 123-4567
================================== Ai Message ==================================

{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}


In [5]:
result.keys()

dict_keys(['messages', 'structured_response'])

In [6]:
print(result["structured_response"])

name='John Doe' email='john@example.com' phone='(555) 123-4567'


In [7]:
result = agent.invoke({
    "messages": [{"role": "human", 
                  "content": "My name is John Doe."}]
})

for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

My name is John Doe.
================================== Ai Message ==================================

{"name":"John Doe","email":"","phone":""}


In [8]:
print(result["structured_response"])

name='John Doe' email='' phone=''


### **Step 4: Agent Init with ToolStrategy for Structured Output**

For models that don’t support native structured output, LangChain uses tool calling to achieve the same result. This works with all models that support tool calling, which is most modern models.

```python
class ToolStrategy(schema):
    schema: schema,
    tool_message_content: str | None,
    handle_errors: ERROR_HANDLING_LOGIC
```

- **schema:** The schema defining the structured output format.
- **tool_message_content:** The tool_message_content parameter allows you to customize the message that appears in the conversation history when structured output is generated as per the given schema.
- **handle_errors:** Error handling strategy for structured output validation failures. By default ERROR_HANDLING_LOGIC is set to boolean value True.

In [9]:
# ! pip install langchain-groq

In [10]:
from langchain_groq import ChatGroq

# Setup API Key
f = open('keys/.groq_api_key.txt')
GROQ_API_KEY = f.read()

groq_chat_model = ChatGroq(
    api_key=GROQ_API_KEY, 
    model="openai/gpt-oss-20b", 
    temperature=1, 
)

In [11]:
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

agent = create_agent(
    model=groq_chat_model,
    response_format=ToolStrategy(schema=Union[ContactInfo,]),  # Default: handle_errors=True
    system_prompt="""You are a helpful assistant. 
                     Be concise and accurate. 
                     You are suppose to extract contact info from the user inputs.
                     """
)

In [12]:
result = agent.invoke({
    "messages": [{"role": "human", 
                  "content": "My name is John Doe, email john@example.com, and number (555) 123-4567"}]
})

for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

My name is John Doe, email john@example.com, and number (555) 123-4567
================================== Ai Message ==================================
Tool Calls:
  ContactInfo (fc_9f118615-6170-4a1e-b7a5-8fec7637a3ee)
 Call ID: fc_9f118615-6170-4a1e-b7a5-8fec7637a3ee
  Args:
    email: john@example.com
    name: John Doe
    phone: (555) 123-4567
================================= Tool Message =================================
Name: ContactInfo

Returning structured response: name='John Doe' email='john@example.com' phone='(555) 123-4567'


In [13]:
print(result["structured_response"])

name='John Doe' email='john@example.com' phone='(555) 123-4567'


In [14]:
agent = create_agent(
    model=groq_chat_model,
    response_format=ToolStrategy(
        schema=Union[ContactInfo,],
        tool_message_content="Output generated as per the given schema."
    ),  # Default: handle_errors=True
    system_prompt="""You are a helpful assistant. 
                     Be concise and accurate. 
                     You are suppose to extract contact info from the user inputs.
                     """
)

In [15]:
result = agent.invoke({
    "messages": [{"role": "human", 
                  "content": "My name is John Doe, email john@example.com, and number (555) 123-4567"}]
})

for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

My name is John Doe, email john@example.com, and number (555) 123-4567
================================== Ai Message ==================================
Tool Calls:
  ContactInfo (fc_1172a177-3fc4-4d49-b1be-94535b7b6a01)
 Call ID: fc_1172a177-3fc4-4d49-b1be-94535b7b6a01
  Args:
    email: john@example.com
    name: John Doe
    phone: (555) 123-4567
================================= Tool Message =================================
Name: ContactInfo

Output generated as per the given schema.


In [16]:
print(result["structured_response"])

name='John Doe' email='john@example.com' phone='(555) 123-4567'


### **Important: Passing Multiple Schemas** 
1. The `Union[ContactInfo,]` enables multiple possible schemas—the model picks the best match (or you get type errors if invalid).
2. Without Union, only one exact schema allowed. `Union[ContactInfo,]` (note trailing comma) is a single-item Union—technically valid Python but redundant here.
3. Agent auto retries if:
    - Wrong schema picked
    - Validation fails (eg: missing email id)

## **Error Handling**

### **Multiple Structured Outputs Error**

Models can make mistakes when generating structured output via tool calling. 

Sometimes cheap llms (like gpt-4o-mini) can greedily calls 2+ schemas. This is rare with good models/prompts.

LangChain provides intelligent retry mechanisms to handle these errors automatically.

In [1]:
from pydantic import BaseModel, Field

class ContactInfo(BaseModel):
    name: str = Field(description="Person's name")
    email: str = Field(description="Email address")

class EventDetails(BaseModel):
    event_name: str = Field(description="Name of the event")
    date: str = Field(description="Event date")

In [18]:
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

agent = create_agent(
    model=openai_chat_model,
    tools=[],
    response_format=ToolStrategy(
        Union[ContactInfo, EventDetails], 
        handle_errors=True
    ),
)

In [19]:
response = agent.invoke({
    "messages": [{"role": "user", 
                  "content": "Extract info: John Doe (john@email.com) is organizing Tech Conference on March 15th"}]
})

for msg in response["messages"]:
    msg.pretty_print()

### **Important Consideration**
This can increase the token consumption.

When a model incorrectly calls multiple structured output tools, the agent provides error feedback in a ToolMessage and prompts the model to retry.

**Output**
```
================================ Human Message =================================

Extract info: John Doe (john@email.com) is organizing Tech Conference on March 15th
None
================================== Ai Message ==================================
Tool Calls:
  ContactInfo (call_1)
 Call ID: call_1
  Args:
    name: John Doe
    email: john@email.com
  EventDetails (call_2)
 Call ID: call_2
  Args:
    event_name: Tech Conference
    date: March 15th
================================= Tool Message =================================
Name: ContactInfo

Error: Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) when only one is expected.
 Please fix your mistakes.
================================= Tool Message =================================
Name: EventDetails

Error: Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) when only one is expected.
 Please fix your mistakes.
================================== Ai Message ==================================
Tool Calls:
  ContactInfo (call_3)
 Call ID: call_3
  Args:
    name: John Doe
    email: john@email.com
================================= Tool Message =================================
Name: ContactInfo

Returning structured response: {'name': 'John Doe', 'email': 'john@email.com'}
```

<img src="images/token_usage.png" width="350" style="float: right; margin-left: 20px; margin-bottom: 10px;">

**Rate Limit Issue** 
- LLM sees multiple tool schemas in its tool list
- It often calls 2+ erroneously (greedy token generation)
- Agent detects violation → injects `ToolMessage` error: `"Model incorrectly returned multiple structured responses... Please fix your mistakes."`
- LLM retries → another call → rate limit hit
<img src="images/ratelimit_error.png">

**Solution** 
1. Refactor Schema Handling instead of juggling multiple schemas, split the process into a parallel workflow. Create two distinct nodes, where each node is responsible for a single response_format.
2. Use a unified schema, better system prompt, and disable retries for Rate Limit Control.
    - Disable Retries for Rate Limit Control: `handle_errors=False` this ensures no retries and fail-fast.
    - Unified Schema: Nest both ContactInfo and EventDetails under a single response object—the LLM fills only relevant fields.
    - Better System Prompt: Explicit instructions reduce multi-calls

### **Solution: Disable Retries for Rate Limit Control**

In [3]:
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

agent = create_agent(
    model=openai_chat_model,
    tools=[],
    response_format=ToolStrategy(
        Union[ContactInfo, EventDetails], 
        handle_errors=False             # No retries, fail-fast
    ),
    system_prompt="""Extract ALL mentioned info into UnifiedResponse.
                    - contacts: ONLY if name+email/phone detected
                    - events: ONLY if event name+date/location detected
                    Populate empty lists [] if type absent. Never use multiple tool calls.""",
)

In [4]:
response = agent.invoke({
    "messages": [{"role": "user", 
                  "content": "Extract info: John Doe (john@email.com) is organizing Tech Conference on March 15th"}]
})

for msg in response["messages"]:
    msg.pretty_print()

MultipleStructuredOutputsError: Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) when only one is expected.

### **Solution: Unified Schema, Disable Retries for Rate Limit Control and Better System Prompt**

In [7]:
from typing import Optional

class UnifiedResponse(BaseModel):
    """Single response containing any detected info - populate only relevant fields"""
    contacts: Optional[list[ContactInfo]] = Field(default=None, description="List detected contacts")
    events: Optional[list[EventDetails]] = Field(default=None, description="List detected events")


agent = create_agent(
    model=openai_chat_model,
    tools=[], 
    response_format=ToolStrategy(UnifiedResponse)  # Single schema = zero retry risk
)

response = agent.invoke({
    "messages": [{"role": "user", 
                  "content": "Extract info: John Doe (john@email.com) is organizing Tech Conference on March 15th"}]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Extract info: John Doe (john@email.com) is organizing Tech Conference on March 15th
================================== Ai Message ==================================
Tool Calls:
  UnifiedResponse (call_RTOAs0LmYICJFcz9A1o1euof)
 Call ID: call_RTOAs0LmYICJFcz9A1o1euof
  Args:
    contacts: [{'name': 'John Doe', 'email': 'john@email.com'}]
    events: [{'event_name': 'Tech Conference', 'date': 'March 15th'}]
================================= Tool Message =================================
Name: UnifiedResponse

Returning structured response: contacts=[ContactInfo(name='John Doe', email='john@email.com')] events=[EventDetails(event_name='Tech Conference', date='March 15th')]


In [10]:
print(response.keys())

dict_keys(['messages', 'structured_response'])


In [11]:
print(response["structured_response"])

contacts=[ContactInfo(name='John Doe', email='john@email.com')] events=[EventDetails(event_name='Tech Conference', date='March 15th')]


### **Schema validation error**

When structured output doesn’t match the expected schema, the agent provides specific error feedback.

In [20]:
from pydantic import BaseModel, Field

class ProductRating(BaseModel):
    rating: int | None = Field(description="Rating from 1-5", ge=1, le=5)
    comment: str = Field(description="Review comment")

In [21]:
agent = create_agent(
    model=openai_chat_model,
    tools=[],
    response_format=ToolStrategy(ProductRating),  # Default: handle_errors=True
    system_prompt="""You are a helpful assistant that parses product reviews and ratings. 
                     Parse EXACTLY as user says. Do NOT fix values. Even if user gives rating >5.
                     Do not make any false field or false value."""
)

In [22]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "User mentioned Amazing product, 10/10!"}]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

User mentioned Amazing product, 10/10!
================================== Ai Message ==================================
Tool Calls:
  ProductRating (call_Fh1C8ZQixfGVDUSie8VDfd7P)
 Call ID: call_Fh1C8ZQixfGVDUSie8VDfd7P
  Args:
    rating: 10
    comment: Amazing product, 10/10!
================================= Tool Message =================================
Name: ProductRating

Error: Failed to parse structured output for tool 'ProductRating': Failed to parse data to ProductRating: 1 validation error for ProductRating
rating
  Input should be less than or equal to 5 [type=less_than_equal, input_value=10, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/less_than_equal.
 Please fix your mistakes.
================================== Ai Message ==================================
Tool Calls:
  ProductRating (call_cQ55XfXrKWhsSGWOsNRuWXTA)
 Call ID: call_cQ55XfXrKWhsSGWOsNRu

### **Important Consideration**

**ISSUE 1:** Some models will intentionally fix 10 to 5 at server-side because they knows your schema constraints (ge=1, le=5) from the description.
> **SOLUTION 1:** Use a strict prompt to enforce model to parse the EXACT values.

**ISSUE 2:** Groq does strict validation server-side. Due to this it will throw 400 error even before LangChain's `handle_error=True` can catch/convert to `ToolMessage` retry. 
> **SOLUTION 2:** We can disable this behaviour by passing `kwargs={"strict": "false"}`.

### **Custom Errors**


In [23]:
from langchain.agents.structured_output import StructuredOutputValidationError
from langchain.agents.structured_output import MultipleStructuredOutputsError

def custom_error_handler(error: Exception) -> str:
    if isinstance(error, StructuredOutputValidationError):
        return "There was an issue with the format. Try again."
    elif isinstance(error, MultipleStructuredOutputsError):
        return "Multiple structured outputs were returned. Pick the most relevant one."
    else:
        return f"Error: {str(error)}"


agent = create_agent(
    model=openai_chat_model,
    tools=[],
    response_format=ToolStrategy(
        schema=ProductRating, 
        handle_errors=custom_error_handler
    ),
    system_prompt="""You are a helpful assistant that parses product reviews and ratings. 
                     Parse EXACTLY as user says. Do NOT fix values. Even if user gives rating >5.
                     Do not make any false field or false value."""
)

In [24]:
response = agent.invoke({
    "messages": [{"role": "user", 
                  "content": "User mentioned Amazing product, 10/10!"}]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

User mentioned Amazing product, 10/10!
================================== Ai Message ==================================
Tool Calls:
  ProductRating (call_8P6MvIc8MAWpcnVYdPgxr24M)
 Call ID: call_8P6MvIc8MAWpcnVYdPgxr24M
  Args:
    rating: 10
    comment: Amazing product, 10/10!
================================= Tool Message =================================
Name: ProductRating

There was an issue with the format. Try again.
================================== Ai Message ==================================
Tool Calls:
  ProductRating (call_d02wyGUh0kQcqqXO3LmYvpQ6)
 Call ID: call_d02wyGUh0kQcqqXO3LmYvpQ6
  Args:
    rating: 10
    comment: Amazing product, 10/10!
================================= Tool Message =================================
Name: ProductRating

There was an issue with the format. Try again.
================================== Ai Message ==================================
Tool Calls:
  Prod

In [28]:
for msg in response['messages']:
    # If message is actually a ToolMessage object (not a dict), check its class name
    if type(msg).__name__ == "ToolMessage":
        print(msg.content)
    # If message is a dictionary or you want a fallback
    elif isinstance(msg, dict) and msg.get('tool_call_id'):
        print(msg['content'])

There was an issue with the format. Try again.
There was an issue with the format. Try again.
Returning structured response: rating=None comment='Amazing product, 10/10!'


In [29]:
print(response.keys())

dict_keys(['messages', 'structured_response'])


In [30]:
print(response["structured_response"])

rating=None comment='Amazing product, 10/10!'
